# Feature studies: compare a replica's features across fits

Loads **three fits** and compares, at the **same epoch** and for the **same
replica**, the **features** -- the eigenvectors $q^{(k)}$ of the matrix $H$
(the perp-projected NTK metric). A feature is a vector over $(\mathrm{flavour},
x)$, exactly like a PDF, so it is plotted value-vs-$x$, one figure per flavour,
with the three fits overlaid.

The features come from the `feature_grids_at_epoch` provider (the $q$ columns of
the NTK decomposition at one epoch); `plot_grids` then overlays them. One
comparison page (set of per-flavour figures) is produced per requested rank.

Note: building the decomposition runs at the chosen `EPOCH` only, but still
computes the NTK eigenvectors for that epoch, so it can be slow on the first call.

In [1]:
%matplotlib inline
from IPython.display import display

from matplotlib import pyplot as plt

# Importing the API first sets KERAS_BACKEND=jax (via ntkpdf/__init__.py).
from ntkpdf.api import API
from ntkpdf.plotting.pdfplots_providers import DEFAULT_FLAVOURS, DEFAULT_YLABELS
from ntkpdf.plotting.pdfplots_utils import plot_grids
from ntkpdf.plotting.style import OKABE_ITO, HandlerSpec, ComposedHandler
from ntkpdf.utils import EVOL_LIST

import logging
logging.basicConfig(level=logging.INFO)

Applying NTKPDF plotting style...
Using Keras backend


Could not import module colibri.blackjax_fit


ModuleNotFoundError: No module named 'blackjax'

In [ ]:
# --- Edit these ---------------------------------------------------------
FITS = [
    "260611-ac-03-ntk-sgd", # gamma = 0.001
    "260611-ac-01-ntk-sgd", # gamma = 1
    "260622-ac-01-ntk-sgd", # gamma = 2
    "260611-ac-02-ntk-sgd", # gamma = 3
    "260622-ac-02-ntk-sgd", # gamma = 4
    "260622-ac-03-ntk-sgd", # gamma = 5
    "260622-ac-03-ntk-sgd-long", # gamma = 5
]
FITS_DICT = {"gamma_001",
             "gamma_1",
             "gamma_2",
             "gamma_3",
             "gamma_4",
             "gamma_5",
             "gamma_5_long",
}
REPLICA = 3            # 1-based replica id (same replica across all fits)
# ------------------------------------------------------------------------

# Minimal data/theory context the FK / NTK chain needs.
common_dict = dict(
    dataset_inputs={"from_": "fit"},
    use_cuts="fromfit",
    theory={"from_": "fit"},
    theoryid={"from_": "theory"},
)

# Fit with grokking, different epochs

In [ ]:
high_loss_features = API.feature_grids_at_epoch(fit=FITS_DICT['gamma_5'],
                                                **common_dict,
                                                name="model",
                                                epoch=5000,
                                                evol_fl_names=list(EVOL_LIST),
                                                training=True,
                                                rank_indices=[1,2,3,4,5,6,7,8,9,10],
                                                replica_index=REPLICA)
low_loss_features = API.feature_grids_at_epoch(fit=FITS_DICT['gamma_5'],
                                                **common_dict,
                                                name="model",
                                                epoch=15000,
                                                evol_fl_names=list(EVOL_LIST),
                                                training=True,
                                                rank_indices=[1,2,3,4,5,6,7,8,9,10],
                                                replica_index=REPLICA)
high_loss_features_no_grokking = API.feature_grids_at_epoch(fit=FITS_DICT['gamma_1'],
                                                **common_dict,
                                                name="model",
                                                epoch=5000,
                                                evol_fl_names=list(EVOL_LIST),
                                                training=True,
                                                rank_indices=[1,2,3,4,5,6,7,8,9,10],
                                                replica_index=REPLICA)
low_loss_features_no_grokking = API.feature_grids_at_epoch(fit=FITS_DICT['gamma_1'],
                                                **common_dict,
                                                name="model",
                                                epoch=15000,
                                                evol_fl_names=list(EVOL_LIST),
                                                training=True,
                                                rank_indices=[1,2,3,4,5,6,7,8,9,10],
                                                replica_index=REPLICA)

In [ ]:
import pathlib

ncols = 2
nrows = -(-len(DEFAULT_FLAVOURS) // ncols)   # ceil
SAVEPATH = pathlib.Path.cwd() / "figs" / "feature_studies"
SAVEPATH.mkdir(parents=True, exist_ok=True)

for r in range(10):
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 2.8 * nrows),
                            sharex=True, constrained_layout=True)


    _ = list(plot_grids(
            [high_loss_features[r], low_loss_features[r],
            high_loss_features_no_grokking[r], low_loss_features_no_grokking[r]],
            flavours=DEFAULT_FLAVOURS,
            ylabels=DEFAULT_YLABELS,
            labels=[rf"Epoch 5000", rf"Epoch 15000", r"$\gamma=5$", r"$\gamma=1$"],
            plot_provider="line",
            yscale="linear",
            xscale="log",
            plot_args=[
              {'color': OKABE_ITO[0]}, 
              {'color': OKABE_ITO[1]},
              {'color': OKABE_ITO[0], "linestyle" : '--'}, 
              {'color': OKABE_ITO[1], "linestyle" : '--'}
            ],
            axes=axes.flatten()
    ))

    # plot_grids draws a legend per panel; collapse them to one shared figure legend.
    for i, ax in enumerate(axes.flatten()):
        lg = ax.get_legend()
        if lg is not None and i != 0:
            lg.remove()
        else:
          ax.legend(
              handles=[
                  HandlerSpec(OKABE_ITO[0], None, '-', False),
                  HandlerSpec(OKABE_ITO[1], None, '-', False),
                  HandlerSpec("black", None, '-', False),
                  HandlerSpec("black", None, '--', False),
              ],
              labels=[r"Epoch 5000", r"Epoch 15000", r"$\gamma=5$", r"$\gamma=1$"],
              handler_map={HandlerSpec: ComposedHandler()},
              loc="upper right",
          )
        if i == 1:
            ax.text(0.7, 0.8, rf"$r={r+1}$", ha="center", va="bottom", transform=ax.transAxes)

    # Save the figure
    fig.savefig(SAVEPATH / f"feature_studies_{r+1}.pdf", bbox_inches="tight")

    # Delete the figure to free memory (especially important in notebooks).
    plt.close(fig)